# All-Model Isolated Embedding Experiment

This notebook generalizes the isolated-word embedding pilot from `02_isolated_embedding_experiment.ipynb` across the three planned transformer models: BERTurk, multilingual BERT, and XLM-R. The pilot notebook remains the BERTurk-only baseline; this notebook keeps the same isolated-word setup while making the model, layer, and pooling choices reusable.

The evaluation target is the AnlamVer human semantic similarity score `Sim`. Each unique word is encoded in isolation, special tokens are excluded, subtoken vectors are pooled into one word vector, and cosine similarities are computed for each pair. Spearman correlation is then used to compare model similarities with human similarity rankings.


## Imports and paths

The raw AnlamVer file is semicolon-delimited and encoded with Turkish Windows encoding, so it is read with `encoding="cp1254"`, `sep=";"`, and comma-decimal parsing.


In [1]:
from pathlib import Path
import gc
import random

import numpy as np
import pandas as pd
import torch
from scipy.stats import spearmanr
from transformers import AutoModel, AutoTokenizer

# Fix random seeds so repeated runs are as reproducible as possible.
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

pd.set_option("display.max_colwidth", 120)

# Keep all file paths relative to the notebook/project root.
PROJECT_ROOT = Path.cwd()
DATA_PATH = PROJECT_ROOT / "data" / "raw" / "anlamver-final.csv"
UNIQUE_WORDS_PATH = PROJECT_ROOT / "outputs" / "tables" / "0105-tokenization_unique_words.csv"
RESULTS_DIR = PROJECT_ROOT / "outputs" / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

PAIR_SIMILARITIES_PATH = RESULTS_DIR / "0301-isolated_all_models_pair_similarities.csv"
SUMMARY_PATH = RESULTS_DIR / "0302-isolated_all_models_summary.csv"

# Compare the same layers and pooling strategies for every model.
LAYERS = [1, 7, 12]
POOLING_STRATEGIES = ["first", "last", "mean", "max", "middle"]

# Prefer Apple Silicon MPS first, then CUDA, then CPU.
if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
elif torch.cuda.is_available():
    DEVICE = torch.device("cuda")
else:
    DEVICE = torch.device("cpu")

print("*" * 30)

print(f"Project root: {PROJECT_ROOT}")
print(f"Device: {DEVICE}")


/Users/nebox/Library/CloudStorage/GoogleDrive-anebieren@gmail.com/My Drive/ORTAK/nebox/Lectures/+2nd semester/Thesis/+thesis-proJect/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


******************************
Project root: /Users/nebox/Library/CloudStorage/GoogleDrive-anebieren@gmail.com/My Drive/ORTAK/nebox/Lectures/+2nd semester/Thesis/+thesis-proJect
Device: mps


## Load data

The pair dataset supplies `W1`, `W2`, and the human similarity score `Sim`. The unique-word table is loaded separately so the notebook computes each isolated word embedding once per model.


In [2]:
def load_anlamver_pairs(path: Path) -> pd.DataFrame:
    # The raw AnlamVer file uses semicolons, Turkish Windows encoding, and comma decimals.
    pairs = pd.read_csv(path, encoding="cp1254", sep=";", decimal=",")
    required_columns = {"W1", "W2", "Sim"}
    missing_columns = required_columns - set(pairs.columns)
    if missing_columns:
        raise ValueError(f"Missing required columns: {sorted(missing_columns)}")

    # Standardize the columns needed for the similarity experiment.
    pairs = pairs.copy()
    pairs["W1"] = pairs["W1"].astype("string").str.strip()
    pairs["W2"] = pairs["W2"].astype("string").str.strip()
    pairs["Sim"] = pd.to_numeric(pairs["Sim"], errors="coerce")
    pairs = pairs.dropna(subset=["W1", "W2", "Sim"]).reset_index(drop=True)
    pairs = pairs[(pairs["W1"] != "") & (pairs["W2"] != "")].reset_index(drop=True)
    return pairs


def load_unique_words(path: Path) -> list[str]:
    # Use the tokenization notebook's unique-word table so each word is embedded once.
    unique_word_df = pd.read_csv(path)
    if "word" not in unique_word_df.columns:
        raise ValueError("Expected a 'word' column in the unique-word table.")
    words = unique_word_df["word"].dropna().astype("string").str.strip()
    words = words[words != ""]
    return sorted(words.unique())


pairs = load_anlamver_pairs(DATA_PATH)
unique_words = load_unique_words(UNIQUE_WORDS_PATH)
pair_words = sorted(pd.concat([pairs["W1"], pairs["W2"]]).unique())
# Fail early if the unique-word file does not cover every word in the pair dataset.
missing_words = sorted(set(pair_words) - set(unique_words))
if missing_words:
    raise ValueError(f"Unique-word table is missing {len(missing_words)} pair words, e.g. {missing_words[:10]}")

print(f"Loaded {len(pairs)} AnlamVer pairs")
print(f"Loaded {len(unique_words)} unique words")
pairs[["QID", "W1", "W2", "Sim"]].head()


Loaded 500 AnlamVer pairs
Loaded 317 unique words


,QID,W1,W2,Sim
0,1,mızrap,barınak,0.166667
1,2,kırmızı,gül,1.166667
2,3,suçlu,şüphe,2.416667
3,4,laikçiler,sekülerizmciler,9.000000
4,5,bitki,zeytin,3.666667


## Model registry

The registry gives each model a stable short key for output tables and the Hugging Face model name used by `AutoTokenizer` and `AutoModel`.


In [3]:
# This is the main difference from the BERTurk pilot: run the same workflow for all planned models.
MODEL_REGISTRY = {
    "BERTurk": "dbmdz/bert-base-turkish-cased",
    "mBERT": "bert-base-multilingual-cased",
    "XLM-R": "xlm-roberta-base",
}

MODEL_REGISTRY


{'BERTurk': 'dbmdz/bert-base-turkish-cased',
 'mBERT': 'bert-base-multilingual-cased',
 'XLM-R': 'xlm-roberta-base'}

## Pooling functions

Pooling is applied only to non-special token positions. For words split into multiple subtokens, `first`, `last`, `mean`, `max`, and `middle` provide alternative subtoken-to-word compositions. For even subtoken counts, `middle` selects the right-middle subtoken.


In [4]:
def pool_first(subtoken_embeddings: torch.Tensor) -> torch.Tensor:
    # Use the first real subtoken as the word representation.
    return subtoken_embeddings[0]


def pool_middle(subtoken_embeddings: torch.Tensor) -> torch.Tensor:
    # Use the central real subtoken; for even counts this selects the right-middle subtoken.
    middle_index = len(subtoken_embeddings) // 2
    return subtoken_embeddings[middle_index]


def pool_last(subtoken_embeddings: torch.Tensor) -> torch.Tensor:
    # Use the last real subtoken as the word representation.
    return subtoken_embeddings[-1]


def pool_mean(subtoken_embeddings: torch.Tensor) -> torch.Tensor:
    # Average all real subtokens into one word vector.
    return subtoken_embeddings.mean(dim=0)


def pool_max(subtoken_embeddings: torch.Tensor) -> torch.Tensor:
    # Take the maximum value across subtokens for each embedding dimension.
    return subtoken_embeddings.max(dim=0).values


POOLING_FUNCTIONS = {
    "first": pool_first,
    "last": pool_last,
    "mean": pool_mean,
    "max": pool_max,
    "middle": pool_middle,
}


## Embedding extraction functions

Layer numbers refer to transformer block outputs: layer `1` is the first transformer layer and layer `12` is the final base-model layer. The embedding lookup layer at hidden-state index `0` is not evaluated.


In [5]:
def load_model_and_tokenizer(model_name: str):
    # Load each Hugging Face model with hidden states so layer 1, 7, and 12 are available.
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name, output_hidden_states=True)
    model.config.output_hidden_states = True
    model.to(DEVICE)
    model.eval()
    return tokenizer, model


def get_non_special_token_indices(input_ids: torch.Tensor, tokenizer) -> list[int]:
    # Exclude model-added tokens such as [CLS], [SEP], <s>, and </s> from pooling.
    token_ids = input_ids[0].detach().cpu().tolist()
    special_mask = tokenizer.get_special_tokens_mask(token_ids, already_has_special_tokens=True)
    return [index for index, is_special in enumerate(special_mask) if not is_special]


def tokenize_isolated_word(word: str, tokenizer) -> list[str]:
    # Return only the actual word subtokens for inspection in the sanity check.
    # Encode one word by itself, then move tensors to the selected device.
    encoded = tokenizer(word, add_special_tokens=True, return_tensors="pt")
    token_ids = encoded["input_ids"][0].tolist()
    non_special_indices = get_non_special_token_indices(encoded["input_ids"], tokenizer)
    tokens = tokenizer.convert_ids_to_tokens(token_ids)
    return [tokens[index] for index in non_special_indices]


@torch.no_grad()
def extract_isolated_word_embeddings(
    word: str,
    tokenizer,
    model,
    layers: list[int],
    pooling_functions: dict[str, callable],
    device: torch.device,
) -> dict[int, dict[str, np.ndarray]]:
    encoded = tokenizer(word, add_special_tokens=True, return_tensors="pt")
    encoded = {key: value.to(device) for key, value in encoded.items()}
    non_special_indices = get_non_special_token_indices(encoded["input_ids"], tokenizer)
    if not non_special_indices:
        raise ValueError(f"No non-special subtokens found for word: {word!r}")

    # Inference returns one hidden-state tensor per layer because output_hidden_states=True.
    outputs = model(**encoded)
    hidden_states = outputs.hidden_states
    max_layer = len(hidden_states) - 1
    invalid_layers = [layer for layer in layers if layer < 1 or layer > max_layer]
    if invalid_layers:
        raise ValueError(f"Invalid layers {invalid_layers}; model provides layers 1..{max_layer}")

    # Store embeddings as embeddings[layer][pooling], one vector per layer-pooling combination.
    word_embeddings = {}
    for layer in layers:
        layer_embeddings = hidden_states[layer][0, non_special_indices, :]
        word_embeddings[layer] = {
            pooling_name: pooling_fn(layer_embeddings).detach().cpu().float().numpy()
            for pooling_name, pooling_fn in pooling_functions.items()
        }
    return word_embeddings


def build_embedding_cache(
    words: list[str],
    tokenizer,
    model,
    layers: list[int],
    pooling_functions: dict[str, callable],
    device: torch.device,
) -> dict[str, dict[int, dict[str, np.ndarray]]]:
    # Cache every unique word once so repeated words in pairs do not trigger repeated inference.
    embeddings = {}
    for index, word in enumerate(words, start=1):
        embeddings[word] = extract_isolated_word_embeddings(
            word=word,
            tokenizer=tokenizer,
            model=model,
            layers=layers,
            pooling_functions=pooling_functions,
            device=device,
        )
        if index % 50 == 0 or index == len(words):
            print(f"Cached {index:>3}/{len(words)} words")
    return embeddings


## Pair similarity and evaluation functions

The detailed output is long format: one row per word pair, model, layer, and pooling strategy. This makes it straightforward to filter, aggregate, or plot later.


In [6]:
def cosine_similarity_np(vec_a: np.ndarray, vec_b: np.ndarray) -> float:
    # Cosine similarity measures how close two word vectors are in embedding space.
    denominator = np.linalg.norm(vec_a) * np.linalg.norm(vec_b)
    if denominator == 0:
        return np.nan
    return float(np.dot(vec_a, vec_b) / denominator)


def compute_pair_similarities(
    pairs: pd.DataFrame,
    embeddings: dict[str, dict[int, dict[str, np.ndarray]]],
    model_key: str,
    layers: list[int],
    pooling_strategies: list[str],
) -> pd.DataFrame:
    # Build a long-format table: one row per pair, model, layer, and pooling strategy.
    id_columns = [column for column in ["QID", "W1", "W2", "Sim"] if column in pairs.columns]
    rows = []
    for _, row in pairs.iterrows():
        word_1 = row["W1"]
        word_2 = row["W2"]
        for layer in layers:
            for pooling in pooling_strategies:
                rows.append(
                    {
                        **{column: row[column] for column in id_columns},
                        "model": model_key,
                        "layer": layer,
                        "pooling": pooling,
                        "cosine_similarity": cosine_similarity_np(
                            embeddings[word_1][layer][pooling],
                            embeddings[word_2][layer][pooling],
                        ),
                    }
                )
    return pd.DataFrame(rows)


def summarize_spearman(pair_similarity_df: pd.DataFrame, human_column: str = "Sim") -> pd.DataFrame:
    # Spearman checks whether model similarities rank pairs similarly to human Sim scores.
    rows = []
    group_columns = ["model", "layer", "pooling"]
    for (model_key, layer, pooling), group in pair_similarity_df.groupby(group_columns, sort=False):
        valid = group[[human_column, "cosine_similarity"]].dropna()
        rho, p_value = spearmanr(valid[human_column], valid["cosine_similarity"])
        rows.append(
            {
                "model": model_key,
                "layer": layer,
                "pooling": pooling,
                "spearman_rho": rho,
                "p_value": p_value,
                "n_pairs": len(valid),
            }
        )
    return pd.DataFrame(rows)


def sort_experiment_results(df: pd.DataFrame) -> pd.DataFrame:
    # Use explicit ordering so saved CSVs are reproducible across runs.
    model_order = {model_key: index for index, model_key in enumerate(MODEL_REGISTRY)}
    pooling_order = {pooling: index for index, pooling in enumerate(POOLING_STRATEGIES)}
    sorted_df = df.assign(
        _model_order=df["model"].map(model_order),
        _pooling_order=df["pooling"].map(pooling_order),
    ).sort_values(["_model_order", "layer", "_pooling_order"])
    sort_columns = ["_model_order", "_pooling_order"]
    if "QID" in sorted_df.columns:
        sorted_df = sorted_df.sort_values(["_model_order", "layer", "_pooling_order", "QID"])
    return sorted_df.drop(columns=sort_columns).reset_index(drop=True)


## Model experiment runner

Each model is loaded with hidden states enabled, used to build one cache over the unique words, evaluated across all pair/layer/pooling combinations, and then released before the next model is loaded.


In [7]:
def run_model_isolated_experiment(
    model_key: str,
    model_name: str,
    pairs: pd.DataFrame,
    unique_words: list[str],
) -> tuple[pd.DataFrame, pd.DataFrame]:
    print("=" * 80)
    print(f"Running {model_key}: {model_name}")
    # A fresh tokenizer/model pair is loaded for the current model key.
    tokenizer, model = load_model_and_tokenizer(model_name)
    print(f"Tokenizer: {type(tokenizer).__name__}")
    print(f"Model: {type(model).__name__}")

    # Extract all unique-word embeddings once before computing pair similarities.
    embeddings = build_embedding_cache(
        words=unique_words,
        tokenizer=tokenizer,
        model=model,
        layers=LAYERS,
        pooling_functions=POOLING_FUNCTIONS,
        device=DEVICE,
    )
    # Reuse cached embeddings to compute cosine similarity for every pair.
    pair_similarity_df = compute_pair_similarities(
        pairs=pairs,
        embeddings=embeddings,
        model_key=model_key,
        layers=LAYERS,
        pooling_strategies=POOLING_STRATEGIES,
    )
    pair_similarity_df["model_name"] = model_name
    # Collapse the pair-level similarities into Spearman results per layer and pooling.
    summary_df = summarize_spearman(pair_similarity_df, human_column="Sim")
    summary_df["model_name"] = model_name

    # Delete large objects before loading the next model to reduce memory pressure.
    del embeddings, model, tokenizer
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    if hasattr(torch, "mps") and hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        torch.mps.empty_cache()

    return pair_similarity_df, summary_df


## Sanity check

This small check loads one model, tokenizes one word, and verifies that extracted vectors have the expected shape and finite values before the full all-model run.


In [8]:
# Run a small check before the full experiment so extraction mistakes are caught early.
SANITY_MODEL_KEY = "XLM-R"
SANITY_WORD = "sekülerizmciler" if "sekülerizmciler" in unique_words else unique_words[0]

# XLM-R is useful here because it uses different special tokens from BERT-style models.
sanity_tokenizer, sanity_model = load_model_and_tokenizer(MODEL_REGISTRY[SANITY_MODEL_KEY])
sanity_tokens = tokenize_isolated_word(SANITY_WORD, sanity_tokenizer)
sanity_embeddings = extract_isolated_word_embeddings(
    word=SANITY_WORD,
    tokenizer=sanity_tokenizer,
    model=sanity_model,
    layers=LAYERS,
    pooling_functions=POOLING_FUNCTIONS,
    device=DEVICE,
)

sanity_rows = []
for layer, pooling_dict in sanity_embeddings.items():
    for pooling_name, vector in pooling_dict.items():
        sanity_rows.append(
            {
                "model": SANITY_MODEL_KEY,
                "word": SANITY_WORD,
                "subtokens": sanity_tokens,
                "layer": layer,
                "pooling": pooling_name,
                "shape": vector.shape,
                "finite_values": bool(np.isfinite(vector).all()),
                "l2_norm": float(np.linalg.norm(vector)),
            }
        )

sanity_df = pd.DataFrame(sanity_rows)
# assert means "stop with an error if this condition is not true."
# Check that all requested layers were extracted.
assert set(sanity_embeddings.keys()) == set(LAYERS)
# Check that every layer has all requested pooling strategies.
assert all(set(sanity_embeddings[layer].keys()) == set(POOLING_STRATEGIES) for layer in LAYERS)
# Check that no vector contains NaN or infinite values.
assert sanity_df["finite_values"].all()

# A vector compared with itself should have cosine similarity very close to 1.
same_word_cosine = cosine_similarity_np(sanity_embeddings[7]["mean"], sanity_embeddings[7]["mean"])
assert np.isclose(same_word_cosine, 1.0, atol=1e-6)

print("*" * 80)
print(f"Word: {SANITY_WORD}")
print(f"Non-special subtokens: {sanity_tokens}")
print(f"Same-vector cosine: {same_word_cosine:.6f}")

# Remove the sanity-check model from memory before the full all-model run.
del sanity_embeddings, sanity_model, sanity_tokenizer
if torch.cuda.is_available():
    torch.cuda.empty_cache()
if hasattr(torch, "mps") and hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    torch.mps.empty_cache()



gc.collect()
sanity_df


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 9173.07it/s]
[transformers] XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


********************************************************************************
Word: sekülerizmciler
Non-special subtokens: ['▁se', 'kül', 'er', 'izm', 'ciler']
Same-vector cosine: 1.000000


,model,word,subtokens,layer,pooling,shape,finite_values,l2_norm
0,XLM-R,sekülerizmciler,"[▁se, kül, er, izm, ciler]",1,first,"(768,)",True,13.911016
1,XLM-R,sekülerizmciler,"[▁se, kül, er, izm, ciler]",1,last,"(768,)",True,13.193266
2,XLM-R,sekülerizmciler,"[▁se, kül, er, izm, ciler]",1,mean,"(768,)",True,9.590219
3,XLM-R,sekülerizmciler,"[▁se, kül, er, izm, ciler]",1,max,"(768,)",True,15.724212
4,XLM-R,sekülerizmciler,"[▁se, kül, er, izm, ciler]",1,middle,"(768,)",True,14.311291
5,XLM-R,sekülerizmciler,"[▁se, kül, er, izm, ciler]",7,first,"(768,)",True,21.078159
6,XLM-R,sekülerizmciler,"[▁se, kül, er, izm, ciler]",7,last,"(768,)",True,22.770857
7,XLM-R,sekülerizmciler,"[▁se, kül, er, izm, ciler]",7,mean,"(768,)",True,20.279058
8,XLM-R,sekülerizmciler,"[▁se, kül, er, izm, ciler]",7,max,"(768,)",True,24.312723
9,XLM-R,sekülerizmciler,"[▁se, kül, er, izm, ciler]",7,middle,"(768,)",True,23.648405


## Run full all-model isolated experiment

This cell performs the full extraction and evaluation for BERTurk, mBERT, and XLM-R. It may take several minutes depending on hardware and whether the models are already cached locally.


In [9]:
# Collect one pair-level table and one summary table from each model run.
all_pair_similarity_frames = []
all_summary_frames = []

# This loop is the all-model generalization of the BERTurk-only pilot workflow.
for model_key, model_name in MODEL_REGISTRY.items():
    pair_similarity_df, summary_df = run_model_isolated_experiment(
        model_key=model_key,
        model_name=model_name,
        pairs=pairs,
        unique_words=unique_words,
    )
    all_pair_similarity_frames.append(pair_similarity_df)
    all_summary_frames.append(summary_df)

# Combine model-specific outputs into final CSV-ready tables.
all_pair_similarities = sort_experiment_results(pd.concat(all_pair_similarity_frames, ignore_index=True))
all_summary = sort_experiment_results(pd.concat(all_summary_frames, ignore_index=True))

print(f"Pair-level rows: {len(all_pair_similarities)}")
print(f"Summary rows: {len(all_summary)}")
all_summary


Running BERTurk: dbmdz/bert-base-turkish-cased


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 8864.62it/s]
[transformers] BertModel LOAD REPORT from: dbmdz/bert-base-turkish-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Tokenizer: BertTokenizer
Model: BertModel
Cached  50/317 words
Cached 100/317 words
Cached 150/317 words
Cached 200/317 words
Cached 250/317 words
Cached 300/317 words
Cached 317/317 words
Running mBERT: bert-base-multilingual-cased


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 8933.03it/s]
[transformers] BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Tokenizer: BertTokenizer
Model: BertModel
Cached  50/317 words
Cached 100/317 words
Cached 150/317 words
Cached 200/317 words
Cached 250/317 words
Cached 300/317 words
Cached 317/317 words
Running XLM-R: xlm-roberta-base


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 9245.10it/s]
[transformers] XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Tokenizer: XLMRobertaTokenizer
Model: XLMRobertaModel
Cached  50/317 words
Cached 100/317 words
Cached 150/317 words
Cached 200/317 words
Cached 250/317 words
Cached 300/317 words
Cached 317/317 words
Pair-level rows: 22500
Summary rows: 45


,model,layer,pooling,spearman_rho,p_value,n_pairs,model_name
0,BERTurk,1,first,0.426694,1.533853e-23,500,dbmdz/bert-base-turkish-cased
1,BERTurk,1,last,0.333949,1.721118e-14,500,dbmdz/bert-base-turkish-cased
2,BERTurk,1,mean,0.325226,8.800176e-14,500,dbmdz/bert-base-turkish-cased
3,BERTurk,1,max,0.232726,1.414801e-07,500,dbmdz/bert-base-turkish-cased
4,BERTurk,1,middle,0.317136,3.816103e-13,500,dbmdz/bert-base-turkish-cased
5,BERTurk,7,first,0.315051,5.531052e-13,500,dbmdz/bert-base-turkish-cased
6,BERTurk,7,last,0.287228,5.951604e-11,500,dbmdz/bert-base-turkish-cased
7,BERTurk,7,mean,0.305104,3.120747e-12,500,dbmdz/bert-base-turkish-cased
8,BERTurk,7,max,0.280261,1.777607e-10,500,dbmdz/bert-base-turkish-cased
9,BERTurk,7,middle,0.275134,3.900678e-10,500,dbmdz/bert-base-turkish-cased


## Save outputs

The pair-level table and summary table are sorted by model, layer, and pooling strategy for reproducible downstream use.


In [10]:
# Save the detailed and summary outputs for later thesis analysis.
all_pair_similarities.to_csv(PAIR_SIMILARITIES_PATH, index=False)
all_summary.to_csv(SUMMARY_PATH, index=False)

print(f"Saved pair-level similarities to: {PAIR_SIMILARITIES_PATH}")
print(f"Saved summary results to: {SUMMARY_PATH}")


Saved pair-level similarities to: /Users/nebox/Library/CloudStorage/GoogleDrive-anebieren@gmail.com/My Drive/ORTAK/nebox/Lectures/+2nd semester/Thesis/+thesis-proJect/outputs/results/0301-isolated_all_models_pair_similarities.csv
Saved summary results to: /Users/nebox/Library/CloudStorage/GoogleDrive-anebieren@gmail.com/My Drive/ORTAK/nebox/Lectures/+2nd semester/Thesis/+thesis-proJect/outputs/results/0302-isolated_all_models_summary.csv


## Best results and interpretation notes

The table below ranks configurations by Spearman correlation. Since this is still an isolated-word experiment, the results should be interpreted as model-internal lexical similarity baselines rather than contextual sentence-level performance.


In [11]:
# Rank all model-layer-pooling combinations by Spearman correlation.
best_results = all_summary.sort_values("spearman_rho", ascending=False).reset_index(drop=True)
best_results.head(15)


,model,layer,pooling,spearman_rho,p_value,n_pairs,model_name
0,BERTurk,1,first,0.426694,1.533853e-23,500,dbmdz/bert-base-turkish-cased
1,BERTurk,12,mean,0.356264,2.077249e-16,500,dbmdz/bert-base-turkish-cased
2,BERTurk,12,first,0.356035,2.177668e-16,500,dbmdz/bert-base-turkish-cased
3,BERTurk,12,last,0.340645,4.745115e-15,500,dbmdz/bert-base-turkish-cased
4,BERTurk,1,last,0.333949,1.721118e-14,500,dbmdz/bert-base-turkish-cased
5,BERTurk,1,mean,0.325226,8.800176e-14,500,dbmdz/bert-base-turkish-cased
6,BERTurk,12,max,0.322338,1.493261e-13,500,dbmdz/bert-base-turkish-cased
7,BERTurk,12,middle,0.321913,1.613322e-13,500,dbmdz/bert-base-turkish-cased
8,BERTurk,1,middle,0.317136,3.816103e-13,500,dbmdz/bert-base-turkish-cased
9,BERTurk,7,first,0.315051,5.531052e-13,500,dbmdz/bert-base-turkish-cased


In [12]:
# Keep the single best configuration for each model for quick comparison.
best_by_model = (
    all_summary.sort_values("spearman_rho", ascending=False)
    .groupby("model", as_index=False)
    .head(1)
    .sort_values("model")
    .reset_index(drop=True)
)

best_by_model


,model,layer,pooling,spearman_rho,p_value,n_pairs,model_name
0,BERTurk,1,first,0.426694,1.533853e-23,500,dbmdz/bert-base-turkish-cased
1,XLM-R,12,middle,0.150528,7.335111e-04,500,xlm-roberta-base
2,mBERT,12,first,0.154553,5.241110e-04,500,bert-base-multilingual-cased


In [13]:
top3_by_model = (
    all_summary.sort_values("spearman_rho", ascending=False)
    .groupby("model", as_index=False)
    .head(3)
    .sort_values(["model", "spearman_rho"], ascending=[True, False])
    .reset_index(drop=True)
)

top3_by_model


,model,layer,pooling,spearman_rho,p_value,n_pairs,model_name
0,BERTurk,1,first,0.426694,1.533853e-23,500,dbmdz/bert-base-turkish-cased
1,BERTurk,12,mean,0.356264,2.077249e-16,500,dbmdz/bert-base-turkish-cased
2,BERTurk,12,first,0.356035,2.177668e-16,500,dbmdz/bert-base-turkish-cased
3,XLM-R,12,middle,0.150528,7.335111e-04,500,xlm-roberta-base
4,XLM-R,1,first,0.121172,6.673702e-03,500,xlm-roberta-base
5,XLM-R,12,first,0.115322,9.855854e-03,500,xlm-roberta-base
6,mBERT,12,first,0.154553,5.241110e-04,500,bert-base-multilingual-cased
7,mBERT,12,middle,0.138869,1.854933e-03,500,bert-base-multilingual-cased
8,mBERT,1,first,0.135995,2.307482e-03,500,bert-base-multilingual-cased
